# Dorsal Forebrain Organoid Validation

This notebook runs the Validation & Benchmarking Agent to assess organoid protocols.

## Workflow Phases

1. **Protocol Review** - Initial review of the proposed protocol
2. **QC Assay Definition** - Define quality control assays and timing
3. **Benchmark Comparison** - Compare to human fetal developmental benchmarks
4. **Off-Target Assessment** - Identify potential off-target differentiation
5. **Validation Experiments** - Propose validation experiments
6. **Final Recommendations** - Synthesize final protocol recommendations

## Setup

In [1]:
import concurrent.futures
import json
from pathlib import Path

from virtual_lab.constants import CONSISTENT_TEMPERATURE, CREATIVE_TEMPERATURE
from virtual_lab.prompts import create_merge_prompt
from virtual_lab.run_meeting import run_meeting
from virtual_lab.utils import load_summaries

# Import our validation-specific constants
from validation_constants import (
    background_prompt,
    organoid_context_prompt,
    num_iterations,
    num_rounds,
    discussions_phase_to_dir,
    principal_investigator,
    validation_specialist,
    developmental_biologist,
    stem_cell_specialist,
    computational_analyst,
    scientific_critic,
    validation_team_members,
    qc_team_members,
    benchmarking_team_members,
    DORSAL_FOREBRAIN_MARKERS,
    OFF_TARGET_MARKERS,
    QC_TIMEPOINTS,
    EXAMPLE_PROTOCOL,
)

# Also import the full ValidationBenchmarkingAgent for direct use
from virtual_lab.agents import VALIDATION_BENCHMARKING_AGENT

In [2]:
# Create discussion directories
for phase_dir in discussions_phase_to_dir.values():
    phase_dir.mkdir(parents=True, exist_ok=True)
    print(f"Created: {phase_dir}")

Created: discussions\protocol_review
Created: discussions\qc_assay_definition
Created: discussions\benchmark_comparison
Created: discussions\off_target_assessment
Created: discussions\validation_experiments
Created: discussions\final_recommendations


## Validation Agent Information

Let's explore what the Validation & Benchmarking Agent provides.

In [3]:
# Display agent information
agent = VALIDATION_BENCHMARKING_AGENT

print("=" * 60)
print("VALIDATION & BENCHMARKING AGENT")
print("=" * 60)
print(f"\nTitle: {agent.title}")
print(f"Interaction Style: {agent.interaction_style.value}")
print(f"\nExpertise: {agent.expertise}")
print(f"\nGoal: {agent.goal}")

VALIDATION & BENCHMARKING AGENT

Title: Validation and Benchmarking Specialist
Interaction Style: skeptical

Expertise: organoid quality control, developmental neurobiology benchmarking, scRNA-seq analysis, and validation of in vitro models against human fetal brain references

Goal: rigorously assess whether proposed organoid protocols will produce biologically accurate dorsal forebrain tissues that faithfully recapitulate human brain development


In [4]:
# Display available tools
print("\n" + "=" * 60)
print("AVAILABLE TOOLS")
print("=" * 60)

for tool in agent.tools:
    print(f"\n{tool.name}")
    print(f"  {tool.description[:80]}...")


AVAILABLE TOOLS

brainspan
  Access BrainSpan human brain transcriptome data for developmental benchmarking. ...

fetal_scrnaseq
  Access fetal brain single-cell RNA-seq reference datasets. Compare organoid cell...

organoid_benchmark
  Access organoid benchmarking resources and reproducibility studies. Compare prot...

proteomics
  Access brain proteomics and secretomics datasets. Compare organoid protein profi...

pathway_activity
  Perform pathway activity scoring and enrichment analysis. Validate developmental...


In [5]:
# Display QC assays
print("\n" + "=" * 60)
print("QC ASSAYS")
print("=" * 60)

for assay in agent.qc_assays:
    print(f"\n{assay.name} ({assay.timing})")
    print(f"  Markers: {', '.join(assay.markers[:5])}...")


QC ASSAYS

Neural Induction Verification (Day 6-10)
  Markers: PAX6, SOX1, SOX2, NES...

Dorsal Forebrain Identity (Day 15-25)
  Markers: FOXG1, EMX1, EMX2, PAX6, LHX2...

Progenitor Organization (Day 30-45)
  Markers: PAX6, SOX2, TBR2/EOMES, phospho-VIMENTIN...

Neurogenesis Progression (Day 45-90)
  Markers: TBR1, CTIP2/BCL11B, SATB2, BRN2/POU3F2, REELIN...

Outer Radial Glia Assessment (Day 60-120)
  Markers: HOPX, FAM107A, PTPRZ1, TNC, SOX2...

Functional Maturation (Day 90-180)
  Markers: synaptic proteins, action potentials, network activity...

Gliogenesis Assessment (Day 120-180+)
  Markers: GFAP, S100B, AQP4, OLIG2, MBP...


In [6]:
# Display developmental benchmarks
print("\n" + "=" * 60)
print("DEVELOPMENTAL BENCHMARKS")
print("=" * 60)

for benchmark in agent.benchmarks:
    print(f"\n{benchmark.name} ({benchmark.stage})")
    print(f"  Reference: {benchmark.reference_dataset}")
    print(f"  Cell Types: {', '.join(benchmark.cell_types[:3])}...")


DEVELOPMENTAL BENCHMARKS

Early Cortical Plate (GW8-10)
  Reference: Nowakowski 2017
  Cell Types: ventricular radial glia, early neurons, Cajal-Retzius cells...

Peak Neurogenesis (GW14-18)
  Reference: Polioudakis 2019
  Cell Types: ventricular radial glia, outer radial glia, intermediate progenitors...

Late Neurogenesis (GW20-26)
  Reference: Trevino 2021
  Cell Types: radial glia (declining), upper layer neurons, interneurons (migrating)...


## Protocol Input

Define the organoid protocol to validate. You can modify this with your own protocol.

In [7]:
# Use the example protocol or define your own
protocol_to_validate = EXAMPLE_PROTOCOL

print("Protocol to Validate:")
print(protocol_to_validate)

Protocol to Validate:

Dorsal Forebrain Organoid Protocol (Draft):

Day 0-6: Neural Induction
- Dissociate hPSCs to single cells
- Embed in Matrigel domes
- Culture in neural induction medium (dual SMAD inhibition)
- SB431542 (10 μM) + LDN193189 (100 nM)

Day 6-15: Dorsal Patterning
- Switch to patterning medium
- Add cyclopamine (1 μM) to inhibit SHH
- Maintain in low-attachment plates

Day 15-45: Early Organoid Expansion
- Transfer to spinning bioreactor
- Neural maintenance medium
- Media changes every 3 days

Day 45+: Maturation
- Continue in bioreactor
- Optional: Add BDNF/NT3 for neuronal maturation
- Long-term culture up to 180+ days



---
## Phase 1: Protocol Review

Initial review of the proposed organoid protocol by the validation team.

In [8]:
# Protocol review prompts
protocol_review_agenda = f"""{background_prompt}

{organoid_context_prompt}

Please review the following proposed organoid protocol and provide initial feedback
on its strengths, weaknesses, and areas that need further validation:

{protocol_to_validate}
"""

protocol_review_questions = (
    "What are the key strengths of this protocol for generating dorsal forebrain organoids?",
    "What are the potential weaknesses or gaps in the protocol?",
    "What aspects of the protocol require rigorous validation?",
    "Are there any obvious risks of off-target differentiation based on the protocol design?",
)

In [9]:
# Protocol review - team discussion
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(
            run_meeting,
            meeting_type="team",
            team_lead=principal_investigator,
            team_members=validation_team_members,
            agenda=protocol_review_agenda,
            agenda_questions=protocol_review_questions,
            save_dir=discussions_phase_to_dir["protocol_review"],
            save_name=f"discussion_{iteration_num + 1}",
            temperature=CREATIVE_TEMPERATURE,
            num_rounds=num_rounds,
        )
        for iteration_num in range(num_iterations)
    ]
    concurrent.futures.wait(futures)

Team:   0%|          | 0/6 [00:00<?, ?it/s]

Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]


Team:   0%|          | 0/6 [00:00<?, ?it/s]



Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]




Team:   0%|          | 0/6 [00:00<?, ?it/s]




Team:  17%|█▋        | 1/6 [00:08<00:40,  8.19s/it]


Team:  17%|█▋        | 1/6 [00:12<01:01, 12.21s/it]


Team:  33%|███▎      | 2/6 [00:17<00:34,  8.68s/it]




Team:  50%|█████     | 3/6 [00:30<00:28,  9.65s/it]




Team:  50%|█████     | 3/6 [00:29<00:29,  9.92s/it]


Team:  50%|█████     | 3/6 [00:33<00:36, 12.00s/it]




Team:  67%|██████▋   | 4/6 [00:41<00:20, 10.23s/it]


Team:  67%|██████▋   | 4/6 [00:43<00:22, 11.16s/it]




Team:  83%|████████▎ | 5/6 [00:51<00:10, 10.07s/it]


Team:   0%|          | 0/6 [00:00<?, ?it/s]




Team: 100%|██████████| 6/6 [01:01<00:00, 10.17s/it]




Rounds (+ Final Round):  33%|███▎      | 1/3 [01:01<02:02, 61.01s/it]




Team:   0%|          | 0/6 [00:00<?, ?it/s]


Team: 100%|█

Input token count: 45,618
Output token count: 5,600
Tool token count: 0
Max token length: 7,567
Cost: $0.17
Time: 2:16


Team:   0%|          | 0/6 [00:13<?, ?it/s]


Rounds (+ Final Round): 100%|██████████| 3/3 [02:17<00:00, 45.98s/it]


Input token count: 45,126
Output token count: 5,817
Tool token count: 0
Max token length: 7,784
Cost: $0.17
Time: 2:24


Team:   0%|          | 0/6 [00:21<?, ?it/s]




Rounds (+ Final Round): 100%|██████████| 3/3 [02:18<00:00, 46.06s/it]


Input token count: 43,301
Output token count: 5,336
Tool token count: 0
Max token length: 7,303
Cost: $0.16
Time: 2:26


In [10]:
# Protocol review - merge discussions
protocol_review_summaries = load_summaries(
    discussion_paths=list(discussions_phase_to_dir["protocol_review"].glob("discussion_*.json"))
)
print(f"Number of summaries: {len(protocol_review_summaries)}")

protocol_review_merge_prompt = create_merge_prompt(
    agenda=protocol_review_agenda,
    agenda_questions=protocol_review_questions,
)

run_meeting(
    meeting_type="individual",
    team_member=principal_investigator,
    summaries=protocol_review_summaries,
    agenda=protocol_review_merge_prompt,
    save_dir=discussions_phase_to_dir["protocol_review"],
    save_name="merged",
    temperature=CONSISTENT_TEMPERATURE,
)

Number of summaries: 3


Rounds (+ Final Round): 100%|██████████| 1/1 [00:15<00:00, 15.71s/it]


Input token count: 3,138
Output token count: 765
Tool token count: 0
Max token length: 3,903
Cost: $0.02
Time: 0:18


UnicodeEncodeError: 'charmap' codec can't encode character '\u03bc' in position 16308: character maps to <undefined>

---
## Phase 2: QC Assay Definition

Define quality control assays, timing, markers, and expected outcomes.

In [11]:
# QC assay definition prompts
qc_assay_agenda = f"""{background_prompt}

Based on the protocol review, please define a comprehensive set of quality control
assays for validating dorsal forebrain organoid differentiation. For each assay,
specify the timing, markers, expected outcomes, and failure indicators.

Consider the following timepoints: {', '.join(f'Day {t}' for t in QC_TIMEPOINTS)}

Key marker categories to consider:
- Neural induction: {', '.join(DORSAL_FOREBRAIN_MARKERS['neural_induction'])}
- Dorsal identity: {', '.join(DORSAL_FOREBRAIN_MARKERS['dorsal_identity'])}
- Radial glia: {', '.join(DORSAL_FOREBRAIN_MARKERS['radial_glia'])}
- Cortical neurons: {', '.join(DORSAL_FOREBRAIN_MARKERS['deep_layer_neurons'] + DORSAL_FOREBRAIN_MARKERS['upper_layer_neurons'])}
"""

qc_assay_questions = (
    "What QC assays should be performed at early timepoints (Day 10-25) and what outcomes indicate success?",
    "What QC assays should be performed during mid-differentiation (Day 45-90) and what outcomes indicate success?",
    "What QC assays should be performed at late timepoints (Day 120+) and what outcomes indicate success?",
    "What quantitative thresholds should be used for each assay (e.g., percentage of cells expressing markers)?",
)

# Load prior summaries
qc_assay_prior_summaries = load_summaries(
    discussion_paths=[discussions_phase_to_dir["protocol_review"] / "merged.json"]
)
print(f"Number of prior summaries: {len(qc_assay_prior_summaries)}")

Number of prior summaries: 1


In [12]:
# QC assay definition - team discussion
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(
            run_meeting,
            meeting_type="team",
            team_lead=principal_investigator,
            team_members=qc_team_members,
            summaries=qc_assay_prior_summaries,
            agenda=qc_assay_agenda,
            agenda_questions=qc_assay_questions,
            save_dir=discussions_phase_to_dir["qc_assay_definition"],
            save_name=f"discussion_{iteration_num + 1}",
            temperature=CREATIVE_TEMPERATURE,
            num_rounds=num_rounds,
        )
        for iteration_num in range(num_iterations)
    ]
    concurrent.futures.wait(futures)

Team:   0%|          | 0/4 [00:00<?, ?it/s]

Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]


Team:   0%|          | 0/4 [00:00<?, ?it/s]



Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]




Team:  25%|██▌       | 1/4 [00:09<00:28,  9.52s/it]




Team:  25%|██▌       | 1/4 [00:09<00:29,  9.79s/it]


Team:  25%|██▌       | 1/4 [00:10<00:30, 10.15s/it]


Team:  50%|█████     | 2/4 [00:19<00:19,  9.54s/it]




Team:  75%|███████▌  | 3/4 [00:29<00:09,  9.89s/it]




Team:  75%|███████▌  | 3/4 [00:30<00:10, 10.16s/it]


Team:  75%|███████▌  | 3/4 [00:30<00:10, 10.31s/it]




Team: 100%|██████████| 4/4 [00:41<00:00, 10.36s/it]




Rounds (+ Final Round):  33%|███▎      | 1/3 [00:41<01:22, 41.44s/it]




Team:   0%|          | 0/4 [00:00<?, ?it/s]


Team: 100%|██████████| 4/4 [00:42<00:00, 10.71s/it]


Rounds (+ Final Round):  33%|███▎      | 1/3 [00:42<01:25, 42.83s/it]


Team:   0%|          | 0/4 [00:00<?, ?it/s]




Team:  25%|██▌       | 1/4 [00:12<00:37, 1

Input token count: 31,102
Output token count: 4,227
Tool token count: 0
Max token length: 6,622
Cost: $0.12
Time: 1:38
Input token count: 30,330
Output token count: 4,063
Tool token count: 0
Max token length: 6,458
Cost: $0.12
Time: 1:38


Team:   0%|          | 0/4 [00:15<?, ?it/s]


Rounds (+ Final Round): 100%|██████████| 3/3 [01:41<00:00, 33.92s/it]


Input token count: 34,257
Output token count: 5,184
Tool token count: 0
Max token length: 7,579
Cost: $0.14
Time: 1:47


In [13]:
# QC assay definition - merge
qc_assay_summaries = load_summaries(
    discussion_paths=list(discussions_phase_to_dir["qc_assay_definition"].glob("discussion_*.json"))
)
print(f"Number of summaries: {len(qc_assay_summaries)}")

qc_assay_merge_prompt = create_merge_prompt(
    agenda=qc_assay_agenda,
    agenda_questions=qc_assay_questions,
)

run_meeting(
    meeting_type="individual",
    team_member=principal_investigator,
    summaries=qc_assay_summaries,
    agenda=qc_assay_merge_prompt,
    save_dir=discussions_phase_to_dir["qc_assay_definition"],
    save_name="merged",
    temperature=CONSISTENT_TEMPERATURE,
)

Number of summaries: 3


Rounds (+ Final Round): 100%|██████████| 1/1 [00:15<00:00, 15.72s/it]


Input token count: 3,181
Output token count: 860
Tool token count: 0
Max token length: 4,041
Cost: $0.02
Time: 0:18


---
## Phase 3: Benchmark Comparison

Compare expected organoid outcomes to human fetal developmental benchmarks.

In [14]:
# Benchmark comparison prompts
benchmark_agenda = f"""{background_prompt}

Now we need to establish how the organoid protocol outcomes will be compared
to human fetal brain developmental benchmarks. Please define specific comparisons
using established reference datasets.

Available reference datasets:
- BrainSpan (human brain transcriptome across development)
- Nowakowski 2017 (fetal cortex scRNA-seq, GW5.85-GW37)
- Polioudakis 2019 (fetal cortex scRNA-seq, GW17-GW18)
- Bhaduri 2020 (fetal cortex scRNA-seq, GW6-GW22)
- Trevino 2021 (fetal cortex scRNA-seq, GW14-GW25)

Consider:
- Transcriptomic correlation analyses
- Cell type proportion comparisons
- Developmental trajectory alignment
- Pathway activity scoring
"""

benchmark_questions = (
    "Which reference datasets should be used for benchmarking organoids at each developmental stage?",
    "What transcriptomic metrics should be used to assess alignment with fetal references?",
    "What cell type proportions are expected at each timepoint based on fetal data?",
    "How should pathway activity be scored and compared to developmental references?",
)

# Load prior summaries
benchmark_prior_summaries = load_summaries(
    discussion_paths=[
        discussions_phase_to_dir["protocol_review"] / "merged.json",
        discussions_phase_to_dir["qc_assay_definition"] / "merged.json",
    ]
)
print(f"Number of prior summaries: {len(benchmark_prior_summaries)}")

Number of prior summaries: 2


In [15]:
# Benchmark comparison - team discussion
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(
            run_meeting,
            meeting_type="team",
            team_lead=principal_investigator,
            team_members=benchmarking_team_members,
            summaries=benchmark_prior_summaries,
            agenda=benchmark_agenda,
            agenda_questions=benchmark_questions,
            save_dir=discussions_phase_to_dir["benchmark_comparison"],
            save_name=f"discussion_{iteration_num + 1}",
            temperature=CREATIVE_TEMPERATURE,
            num_rounds=num_rounds,
        )
        for iteration_num in range(num_iterations)
    ]
    concurrent.futures.wait(futures)

Team:   0%|          | 0/4 [00:00<?, ?it/s]

Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]


Team:   0%|          | 0/4 [00:00<?, ?it/s]



Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]




Team:   0%|          | 0/4 [00:00<?, ?it/s]




Team:  25%|██▌       | 1/4 [00:09<00:29,  9.72s/it]


Team:  25%|██▌       | 1/4 [00:11<00:33, 11.25s/it]


Team:  50%|█████     | 2/4 [00:23<00:23, 11.58s/it]




Team:  75%|███████▌  | 3/4 [00:32<00:10, 10.47s/it]


Team:  75%|███████▌  | 3/4 [00:33<00:11, 11.57s/it]




Team:  75%|███████▌  | 3/4 [00:36<00:12, 12.51s/it]


Team: 100%|██████████| 4/4 [00:44<00:00, 11.19s/it]


Rounds (+ Final Round):  33%|███▎      | 1/3 [00:44<01:29, 44.76s/it]


Team:   0%|          | 0/4 [00:00<?, ?it/s]




Team: 100%|██████████| 4/4 [00:45<00:00, 11.43s/it]




Rounds (+ Final Round):  33%|███▎      | 1/3 [00:45<01:31, 45.71s/it]




Team:   0%|          | 0/4 [00:00<?, ?it/s]




Team:  25%|██▌       | 1/4 [00:11<00:33, 11.13s/it

Input token count: 37,767
Output token count: 4,081
Tool token count: 0
Max token length: 7,232
Cost: $0.14
Time: 1:49



Team:   0%|          | 0/4 [00:18<?, ?it/s]


Rounds (+ Final Round): 100%|██████████| 3/3 [01:47<00:00, 35.77s/it]


Input token count: 36,132
Output token count: 3,870
Tool token count: 0
Max token length: 7,021
Cost: $0.13
Time: 1:52


Rounds (+ Final Round): 100%|██████████| 3/3 [02:03<00:00, 41.18s/it]


Input token count: 37,240
Output token count: 4,218
Tool token count: 0
Max token length: 7,369
Cost: $0.14
Time: 2:08


In [16]:
# Benchmark comparison - merge
benchmark_summaries = load_summaries(
    discussion_paths=list(discussions_phase_to_dir["benchmark_comparison"].glob("discussion_*.json"))
)
print(f"Number of summaries: {len(benchmark_summaries)}")

benchmark_merge_prompt = create_merge_prompt(
    agenda=benchmark_agenda,
    agenda_questions=benchmark_questions,
)

run_meeting(
    meeting_type="individual",
    team_member=principal_investigator,
    summaries=benchmark_summaries,
    agenda=benchmark_merge_prompt,
    save_dir=discussions_phase_to_dir["benchmark_comparison"],
    save_name="merged",
    temperature=CONSISTENT_TEMPERATURE,
)

Number of summaries: 3


Rounds (+ Final Round): 100%|██████████| 1/1 [00:13<00:00, 13.93s/it]


Input token count: 3,154
Output token count: 851
Tool token count: 0
Max token length: 4,005
Cost: $0.02
Time: 0:17


---
## Phase 4: Off-Target Assessment

Identify potential off-target fates and how to detect them.

In [17]:
# Off-target assessment prompts
off_target_agenda = f"""{background_prompt}

We need to identify potential off-target differentiation fates and establish
how to detect and quantify them. This is critical for ensuring the organoids
contain primarily dorsal forebrain cell types.

Key off-target signatures to monitor:
- Ventral forebrain: {', '.join(OFF_TARGET_MARKERS['ventral_forebrain'])}
- Midbrain: {', '.join(OFF_TARGET_MARKERS['midbrain'])}
- Hindbrain: {', '.join(OFF_TARGET_MARKERS['hindbrain'])}
- Non-neural: {', '.join(OFF_TARGET_MARKERS['non_neural'])}
- Stress markers: {', '.join(OFF_TARGET_MARKERS['stress_markers'])}

Consider protocol-specific risks and how the protocol design might lead to
off-target differentiation.
"""

off_target_questions = (
    "What are the most likely off-target fates for this specific protocol and why?",
    "At what timepoints should off-target markers be assessed?",
    "What percentage of off-target cells is acceptable vs. concerning?",
    "How can the protocol be modified to reduce off-target differentiation risk?",
)

# Load prior summaries
off_target_prior_summaries = load_summaries(
    discussion_paths=[
        discussions_phase_to_dir["protocol_review"] / "merged.json",
        discussions_phase_to_dir["qc_assay_definition"] / "merged.json",
    ]
)
print(f"Number of prior summaries: {len(off_target_prior_summaries)}")

Number of prior summaries: 2


In [18]:
# Off-target assessment - team discussion
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(
            run_meeting,
            meeting_type="team",
            team_lead=principal_investigator,
            team_members=validation_team_members,
            summaries=off_target_prior_summaries,
            agenda=off_target_agenda,
            agenda_questions=off_target_questions,
            save_dir=discussions_phase_to_dir["off_target_assessment"],
            save_name=f"discussion_{iteration_num + 1}",
            temperature=CREATIVE_TEMPERATURE,
            num_rounds=num_rounds,
        )
        for iteration_num in range(num_iterations)
    ]
    concurrent.futures.wait(futures)

Team:   0%|          | 0/6 [00:00<?, ?it/s]

Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]


Team:   0%|          | 0/6 [00:00<?, ?it/s]



Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]




Team:  17%|█▋        | 1/6 [00:10<00:52, 10.46s/it]


Team:  17%|█▋        | 1/6 [00:11<00:55, 11.01s/it]




Team:  17%|█▋        | 1/6 [00:11<00:58, 11.64s/it]




Team:  33%|███▎      | 2/6 [00:21<00:42, 10.71s/it]


Team:  33%|███▎      | 2/6 [00:24<00:50, 12.60s/it]




Team:  50%|█████     | 3/6 [00:34<00:35, 11.79s/it]


Team:  67%|██████▋   | 4/6 [00:42<00:20, 10.36s/it]




Team:  83%|████████▎ | 5/6 [00:52<00:10, 10.08s/it]


Team:  67%|██████▋   | 4/6 [00:52<00:26, 13.08s/it]




Team:   0%|          | 0/6 [00:00<?, ?it/s]


Team:  83%|████████▎ | 5/6 [01:02<00:12, 12.02s/it]




Team: 100%|██████████| 6/6 [01:08<00:00, 11.34s/it]




Rounds (+ Final Round):  33%|███▎      | 1/3 [01:08<02:16, 68.05s/it]




Team:  17%|█▋        | 1/6 [00:13<01:09, 13.98s/it

Input token count: 62,992
Output token count: 5,281
Tool token count: 0
Max token length: 8,645
Cost: $0.21
Time: 2:29


Team:   0%|          | 0/6 [00:17<?, ?it/s]




Rounds (+ Final Round): 100%|██████████| 3/3 [02:30<00:00, 50.12s/it]


Input token count: 61,874
Output token count: 5,283
Tool token count: 0
Max token length: 8,647
Cost: $0.21
Time: 2:37


Team:   0%|          | 0/6 [00:14<?, ?it/s]


Rounds (+ Final Round): 100%|██████████| 3/3 [02:34<00:00, 51.47s/it]


Input token count: 64,832
Output token count: 5,628
Tool token count: 0
Max token length: 8,992
Cost: $0.22
Time: 2:40


In [19]:
# Off-target assessment - merge
off_target_summaries = load_summaries(
    discussion_paths=list(discussions_phase_to_dir["off_target_assessment"].glob("discussion_*.json"))
)
print(f"Number of summaries: {len(off_target_summaries)}")

off_target_merge_prompt = create_merge_prompt(
    agenda=off_target_agenda,
    agenda_questions=off_target_questions,
)

run_meeting(
    meeting_type="individual",
    team_member=principal_investigator,
    summaries=off_target_summaries,
    agenda=off_target_merge_prompt,
    save_dir=discussions_phase_to_dir["off_target_assessment"],
    save_name="merged",
    temperature=CONSISTENT_TEMPERATURE,
)

Number of summaries: 3


Rounds (+ Final Round): 100%|██████████| 1/1 [00:16<00:00, 16.29s/it]


Input token count: 3,204
Output token count: 656
Tool token count: 0
Max token length: 3,860
Cost: $0.01
Time: 0:19


---
## Phase 5: Validation Experiments

Propose specific validation experiments to verify protocol fidelity.

In [20]:
# Validation experiments prompts
validation_exp_agenda = f"""{background_prompt}

Based on the QC assay definitions, benchmarking criteria, and off-target assessment,
please propose a comprehensive set of validation experiments to verify that the
organoid protocol produces biologically accurate dorsal forebrain organoids.

Consider the following experimental approaches:
- Immunofluorescence / immunohistochemistry
- Flow cytometry
- Single-cell RNA sequencing
- Bulk RNA sequencing
- Spatial transcriptomics
- Electrophysiology (patch clamp, MEA)
- Calcium imaging
- Morphological analysis

For each experiment, specify timing, expected results, and decision criteria.
"""

validation_exp_questions = (
    "What validation experiments are essential vs. optional for protocol acceptance?",
    "What is the recommended order and timing of validation experiments?",
    "What are the specific success criteria for each validation experiment?",
    "How should results be documented and reported for reproducibility?",
)

# Load all prior summaries
validation_exp_prior_summaries = load_summaries(
    discussion_paths=[
        discussions_phase_to_dir["protocol_review"] / "merged.json",
        discussions_phase_to_dir["qc_assay_definition"] / "merged.json",
        discussions_phase_to_dir["benchmark_comparison"] / "merged.json",
        discussions_phase_to_dir["off_target_assessment"] / "merged.json",
    ]
)
print(f"Number of prior summaries: {len(validation_exp_prior_summaries)}")

Number of prior summaries: 4


In [21]:
# Validation experiments - team discussion
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(
            run_meeting,
            meeting_type="team",
            team_lead=principal_investigator,
            team_members=validation_team_members,
            summaries=validation_exp_prior_summaries,
            agenda=validation_exp_agenda,
            agenda_questions=validation_exp_questions,
            save_dir=discussions_phase_to_dir["validation_experiments"],
            save_name=f"discussion_{iteration_num + 1}",
            temperature=CREATIVE_TEMPERATURE,
            num_rounds=num_rounds,
        )
        for iteration_num in range(num_iterations)
    ]
    concurrent.futures.wait(futures)

Team:   0%|          | 0/6 [00:00<?, ?it/s]

Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]


Team:   0%|          | 0/6 [00:00<?, ?it/s]



Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]




Team:  17%|█▋        | 1/6 [00:09<00:47,  9.48s/it]


Team:  17%|█▋        | 1/6 [00:10<00:54, 10.94s/it]




Team:  33%|███▎      | 2/6 [00:20<00:42, 10.55s/it]


Team:  33%|███▎      | 2/6 [00:21<00:43, 10.88s/it]




Team:  50%|█████     | 3/6 [00:29<00:29,  9.94s/it]


Team:  50%|█████     | 3/6 [00:34<00:34, 11.65s/it]




Team:  67%|██████▋   | 4/6 [00:43<00:22, 11.45s/it]


Team:  67%|██████▋   | 4/6 [00:45<00:22, 11.33s/it]




Team:  83%|████████▎ | 5/6 [00:54<00:11, 11.25s/it]


Team:  83%|████████▎ | 5/6 [00:57<00:11, 11.66s/it]




Team:   0%|          | 0/6 [00:00<?, ?it/s]


Team: 100%|██████████| 6/6 [01:08<00:00, 11.34s/it]


Rounds (+ Final Round):  33%|███▎      | 1/3 [01:08<02:16, 68.06s/it]


Team:   0%|          | 0/6 [00:00<?, ?it/s]




Team: 10

Input token count: 86,904
Output token count: 6,053
Tool token count: 0
Max token length: 10,866
Cost: $0.28
Time: 2:34


Rounds (+ Final Round): 100%|██████████| 3/3 [02:30<00:00, 50.08s/it]


Input token count: 85,228
Output token count: 5,658
Tool token count: 0
Max token length: 10,471
Cost: $0.27
Time: 2:36


Team:   0%|          | 0/6 [00:19<?, ?it/s]




Rounds (+ Final Round): 100%|██████████| 3/3 [02:32<00:00, 50.75s/it]


Input token count: 88,004
Output token count: 5,825
Tool token count: 0
Max token length: 10,638
Cost: $0.28
Time: 2:39


In [22]:
# Validation experiments - merge
validation_exp_summaries = load_summaries(
    discussion_paths=list(discussions_phase_to_dir["validation_experiments"].glob("discussion_*.json"))
)
print(f"Number of summaries: {len(validation_exp_summaries)}")

validation_exp_merge_prompt = create_merge_prompt(
    agenda=validation_exp_agenda,
    agenda_questions=validation_exp_questions,
)

run_meeting(
    meeting_type="individual",
    team_member=principal_investigator,
    summaries=validation_exp_summaries,
    agenda=validation_exp_merge_prompt,
    save_dir=discussions_phase_to_dir["validation_experiments"],
    save_name="merged",
    temperature=CONSISTENT_TEMPERATURE,
)

Number of summaries: 3


Rounds (+ Final Round): 100%|██████████| 1/1 [00:12<00:00, 12.23s/it]


Input token count: 3,258
Output token count: 845
Tool token count: 0
Max token length: 4,103
Cost: $0.02
Time: 0:15


---
## Phase 6: Final Recommendations

Synthesize all findings into final protocol recommendations.

In [23]:
# Final recommendations prompts
final_agenda = f"""{background_prompt}

Based on all the previous discussions, please provide final recommendations for:
1. Protocol modifications (if any) to improve reproducibility and fidelity
2. A complete QC checklist with pass/fail criteria
3. Minimum validation experiments required before publication
4. Documentation requirements for reproducibility
5. Known limitations and caveats of the protocol

The goal is to produce a validated protocol that can be confidently used for
downstream applications such as disease modeling and drug screening.
"""

final_questions = (
    "What specific modifications to the protocol are recommended?",
    "What is the complete QC checklist with quantitative pass/fail criteria?",
    "What is the minimum set of validation experiments required?",
    "What are the known limitations and appropriate use cases for organoids from this protocol?",
)

# Load all prior summaries
final_prior_summaries = load_summaries(
    discussion_paths=[
        discussions_phase_to_dir["protocol_review"] / "merged.json",
        discussions_phase_to_dir["qc_assay_definition"] / "merged.json",
        discussions_phase_to_dir["benchmark_comparison"] / "merged.json",
        discussions_phase_to_dir["off_target_assessment"] / "merged.json",
        discussions_phase_to_dir["validation_experiments"] / "merged.json",
    ]
)
print(f"Number of prior summaries: {len(final_prior_summaries)}")

Number of prior summaries: 5


In [24]:
# Final recommendations - team discussion
with concurrent.futures.ThreadPoolExecutor() as executor:
    futures = [
        executor.submit(
            run_meeting,
            meeting_type="team",
            team_lead=principal_investigator,
            team_members=validation_team_members,
            summaries=final_prior_summaries,
            agenda=final_agenda,
            agenda_questions=final_questions,
            save_dir=discussions_phase_to_dir["final_recommendations"],
            save_name=f"discussion_{iteration_num + 1}",
            temperature=CREATIVE_TEMPERATURE,
            num_rounds=num_rounds,
        )
        for iteration_num in range(num_iterations)
    ]
    concurrent.futures.wait(futures)

Team:   0%|          | 0/6 [00:00<?, ?it/s]

Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]


Team:   0%|          | 0/6 [00:00<?, ?it/s]



Rounds (+ Final Round):   0%|          | 0/3 [00:00<?, ?it/s]




Team:   0%|          | 0/6 [00:00<?, ?it/s]


Team:  17%|█▋        | 1/6 [00:13<01:09, 13.81s/it]




Team:  17%|█▋        | 1/6 [00:20<01:40, 20.02s/it]


Team:  33%|███▎      | 2/6 [00:24<00:48, 12.12s/it]




Team:  50%|█████     | 3/6 [00:37<00:36, 12.21s/it]


Team:  67%|██████▋   | 4/6 [00:46<00:22, 11.13s/it]




Team:  50%|█████     | 3/6 [00:48<00:47, 15.90s/it]


Team:  83%|████████▎ | 5/6 [00:54<00:10, 10.03s/it]


Team:  83%|████████▎ | 5/6 [00:56<00:10, 10.44s/it]




Team:   0%|          | 0/6 [00:00<?, ?it/s]




Team:  83%|████████▎ | 5/6 [01:08<00:12, 12.26s/it]


Team: 100%|██████████| 6/6 [01:09<00:00, 11.51s/it]


Rounds (+ Final Round):  33%|███▎      | 1/3 [01:09<02:18, 69.03s/it]


Team:   0%|          | 0/6 [00:00<?, ?it/s]




Team: 100%|█████

Input token count: 95,167
Output token count: 5,466
Tool token count: 0
Max token length: 11,099
Cost: $0.29
Time: 2:32


Rounds (+ Final Round): 100%|██████████| 3/3 [02:30<00:00, 50.12s/it]


Input token count: 94,650
Output token count: 5,431
Tool token count: 0
Max token length: 11,064
Cost: $0.29
Time: 2:36


Team:   0%|          | 0/6 [00:20<?, ?it/s]




Rounds (+ Final Round): 100%|██████████| 3/3 [02:42<00:00, 54.32s/it]


Input token count: 102,794
Output token count: 6,659
Tool token count: 0
Max token length: 12,292
Cost: $0.32
Time: 2:49


In [25]:
# Final recommendations - merge
final_summaries = load_summaries(
    discussion_paths=list(discussions_phase_to_dir["final_recommendations"].glob("discussion_*.json"))
)
print(f"Number of summaries: {len(final_summaries)}")

final_merge_prompt = create_merge_prompt(
    agenda=final_agenda,
    agenda_questions=final_questions,
)

run_meeting(
    meeting_type="individual",
    team_member=principal_investigator,
    summaries=final_summaries,
    agenda=final_merge_prompt,
    save_dir=discussions_phase_to_dir["final_recommendations"],
    save_name="merged",
    temperature=CONSISTENT_TEMPERATURE,
)

Number of summaries: 3


Rounds (+ Final Round): 100%|██████████| 1/1 [00:14<00:00, 14.36s/it]


Input token count: 3,069
Output token count: 706
Tool token count: 0
Max token length: 3,775
Cost: $0.01
Time: 0:17


---
## Results Analysis

Analyze the validation workflow outputs.

In [26]:
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({'font.size': 14})

figure_dir = Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)

In [27]:
# Count words written by each agent across all phases
phase_to_agent_to_word_count = {}

for phase_name in discussions_phase_to_dir:
    phase_dir = discussions_phase_to_dir[phase_name]
    
    if not phase_dir.exists():
        continue
    
    agent_to_text = {}
    
    for path in phase_dir.glob("*.json"):
        with open(path) as f:
            discussion = json.load(f)
        
        for message in discussion:
            agent = message.get("agent", "Unknown")
            if agent == "User":
                continue
            agent_to_text.setdefault(agent, []).append(message.get("message", ""))
    
    phase_to_agent_to_word_count[phase_name] = {}
    for agent, texts in agent_to_text.items():
        word_count = len(" ".join(texts).split())
        phase_to_agent_to_word_count[phase_name][agent] = word_count

# Print summary
for phase, agents in phase_to_agent_to_word_count.items():
    print(f"\nPhase: {phase}")
    for agent, count in agents.items():
        print(f"  {agent}: {count:,} words")


Phase: protocol_review
  Principal Investigator: 4,318 words
  Validation and Benchmarking Specialist: 1,691 words
  Developmental Biologist: 1,779 words
  Stem Cell Specialist: 1,724 words
  Computational Biologist: 1,667 words
  Scientific Critic: 1,861 words

Phase: qc_assay_definition
  Principal Investigator: 4,479 words
  Validation and Benchmarking Specialist: 1,787 words
  Developmental Biologist: 1,891 words
  Scientific Critic: 1,968 words

Phase: benchmark_comparison
  Principal Investigator: 4,635 words
  Validation and Benchmarking Specialist: 1,758 words
  Computational Biologist: 1,664 words
  Scientific Critic: 1,790 words

Phase: off_target_assessment
  Principal Investigator: 4,598 words
  Validation and Benchmarking Specialist: 1,505 words
  Developmental Biologist: 1,475 words
  Stem Cell Specialist: 1,574 words
  Computational Biologist: 1,573 words
  Scientific Critic: 1,656 words

Phase: validation_experiments
  Principal Investigator: 4,866 words
  Validation a

In [28]:
# Load and display the final recommendations
final_merged_path = discussions_phase_to_dir["final_recommendations"] / "merged.json"

if final_merged_path.exists():
    with open(final_merged_path) as f:
        final_discussion = json.load(f)
    
    # Get the last message (summary)
    if final_discussion:
        final_summary = final_discussion[-1].get("message", "")
        print("=" * 60)
        print("FINAL RECOMMENDATIONS SUMMARY")
        print("=" * 60)
        print(final_summary[:3000] + "..." if len(final_summary) > 3000 else final_summary)
else:
    print("Final recommendations not yet generated. Run all phases first.")

FINAL RECOMMENDATIONS SUMMARY
### Final Recommendations for Dorsal Forebrain Organoid Protocol

Based on the comprehensive input from previous meetings, here are the consolidated recommendations for developing dorsal forebrain organoids that accurately model human cortical development:

#### 1. Protocol Modifications

**Recommended Modifications:**
- **Transition to Synthetic Matrices:** Implement synthetic matrices such as PEG or hyaluronic acid-based hydrogels to reduce variability and enhance reproducibility. This recommendation is consistent across all summaries, emphasizing its importance in improving protocol fidelity.
- **Incorporate Additional Signaling Modulators:** Use WNT and BMP inhibitors to enhance dorsal patterning. This was highlighted in all summaries as a critical step to ensure accurate developmental cues.
- **Optimize Timing of Modulator Application:** Align the timing of signaling molecule exposure with natural developmental processes, as suggested by the Developme

---
## Direct Agent Usage (Bonus)

You can also use the ValidationBenchmarkingAgent directly for quick checks.

In [29]:
# Example: Check for off-target markers in a gene list
sample_genes = [
    "PAX6", "SOX2", "FOXG1", "EMX1",  # Good - dorsal forebrain
    "NKX2.1",                           # Bad - ventral forebrain
    "ATF4", "DDIT3",                    # Concerning - stress markers
]

off_target_issues = VALIDATION_BENCHMARKING_AGENT.check_off_target_fates(sample_genes)

print("Off-target fate analysis:")
if off_target_issues:
    for fate, markers in off_target_issues.items():
        print(f"  WARNING - {fate}: {markers}")
else:
    print("  No off-target markers detected")

Off-target fate analysis:
  WARNING - ventral_forebrain: ['NKX2.1']
  WARNING - stressed_cells: ['ATF4', 'DDIT3']


In [30]:
# Example: Generate a validation protocol for specific timepoints
protocol = VALIDATION_BENCHMARKING_AGENT.generate_validation_protocol(
    timepoints=[25, 60, 120]
)

print("Generated Validation Protocol:")
print("=" * 40)

for timepoint, assays in protocol["timepoints"].items():
    print(f"\n{timepoint}:")
    for assay in assays:
        print(f"  - {assay['assay']}")
        print(f"    Markers: {', '.join(assay['markers'][:4])}...")

Generated Validation Protocol:

Day 25:
  - Dorsal Forebrain Identity
    Markers: FOXG1, EMX1, EMX2, PAX6...

Day 60:
  - Neurogenesis Progression
    Markers: TBR1, CTIP2/BCL11B, SATB2, BRN2/POU3F2...
  - Outer Radial Glia Assessment
    Markers: HOPX, FAM107A, PTPRZ1, TNC...

Day 120:
  - Outer Radial Glia Assessment
    Markers: HOPX, FAM107A, PTPRZ1, TNC...
  - Functional Maturation
    Markers: synaptic proteins, action potentials, network activity...


In [31]:
# Example: Evaluate scRNA-seq cell type alignment
sample_organoid_composition = {
    "ventricular radial glia": 0.20,
    "outer radial glia": 0.15,
    "intermediate progenitors": 0.10,
    "deep layer neurons": 0.25,
    "upper layer neurons": 0.15,
    "astrocyte precursors": 0.05,
    "unknown": 0.10,
}

alignment = VALIDATION_BENCHMARKING_AGENT.evaluate_scrnaseq_alignment(
    organoid_cell_types=sample_organoid_composition,
    reference_stage="GW14-18"
)

print("scRNA-seq Alignment Evaluation:")
print("=" * 40)
print(f"Reference: {alignment['reference_stage']}")
print(f"\nMissing cell types: {alignment['missing_types']}")
print(f"Unexpected types: {alignment['unexpected_types']}")
print(f"\nConcerns:")
for concern in alignment['concerns']:
    print(f"  - {concern}")

scRNA-seq Alignment Evaluation:
Reference: GW14-18

Missing cell types: ['upper layer neurons (emerging)']
Unexpected types: ['astrocyte precursors', 'unknown', 'upper layer neurons']

Concerns:
  - Missing expected cell types: upper layer neurons (emerging). Consider extending culture time or adjusting protocol.
  - Unexpected cell types detected: astrocyte precursors, unknown, upper layer neurons. May indicate off-target differentiation.
